# **COMFYUI SERVER NOTES**

This notebook loads ComfyUI in a headless mode and implements a hard-coded **text to image** workflow. It is designed to load a **UNET Model** and up to 2 LoRAs. Smaller models are a bit more stable as this Colab notebook has limitation on the Google Free tier. Any models and LoRAs that are used need to fit into those limitations.
* RAM: 12.7GB - VRAM: 15GB - DISK: 112GB

## TESTED MODELS
| Model | Quantized | Notes |
|-------|--------|--------|
| Z-Image-Turbo | FP8 | fast - ALWAYS use the FIXED version of ComfyUI |
| Z-Image-Base | FP8 | ok |
| Anima Preview | FP8 | ok |
| Flux.2 9B Klein | FP8 | ok - you can gen about 4-5 images before crashing|
| Qwen Image | GGUF | very slow - only kept it as an example on loading gguf |


In [ ]:
#@title #Step 1: Install Comfy UI
from IPython.display import Audio, display, clear_output
from IPython.utils import capture
import os, time
t0 = time.time()

#@markdown ##ComfyUI Install Options
INSTALL_IN_GDRIVE = False # @param {type:"boolean"}
INSTALL_GGUF = False # @param {type:"boolean"}
#@markdown * **INSTALL_IN_DRIVE**: You can install ComfyUI to your GDrive or to the local runtime filesystem.
#@markdown * **INSTALL_GGUF**: You only need to install GGUF if you are loading a GGUF (Qwen) model.
COMFY_VERSION = "Fixed Version" # @param ["Latest Version", "Fixed Version", "Use Custom Version"]
CUSTOM_VERSION = ""  # @param {type:"string"}
#@markdown * **COMFY_VERSION**: Always select **Latest Version** unless you are using Z-Image-Turbo.
#@markdown   * Z-Image-Turbo runs about 10x faster with the **Fixed Version**.
#@markdown * **CUSTOM_VERSION**: You can select a custom version of ComfyUI if you need to.

if INSTALL_IN_GDRIVE:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    BASE_DIR = "/content/drive/MyDrive/ComfyUI"
else:
    BASE_DIR = "/content/ComfyUI"

# Create directory if needed
if not os.path.exists(BASE_DIR):
    print("📥 Cloning ComfyUI...")
    %cd /content
    !git clone https://github.com/comfyanonymous/ComfyUI "{BASE_DIR}"
    print(f"✅ ComfyUI cloned to {BASE_DIR}.")
else:
    print("✅ ComfyUI already exists, skipping clone.")

%cd "{BASE_DIR}"
if COMFY_VERSION != "Latest Version":
    if COMFY_VERSION == "Fixed Version":
        # This fixed version works best with z-image-turbo
        # I'm not quite sure why but latest versions are very slow
        VERSION = "3c8456223c5f6a41af7d99219b391c8c58acb552"
        print(f"Using ComfyUI Fixed Version: {VERSION}.")
    elif COMFY_VERSION == "Use Custom Version":
        if not CUSTOM_VERSION.strip():
            raise ValueError("CUSTOM_VERSION is empty. Please provide a commit hash or tag.")
        VERSION = CUSTOM_VERSION
        print(f"Using ComfyUI Custom Version: {VERSION}.")

    # Get current commit
    import subprocess
    current_commit = subprocess.check_output(
        ["git", "rev-parse", "HEAD"]
    ).decode().strip()

    if current_commit.startswith(VERSION[:7]):
        print(f"ComfyUI already at {VERSION}.")
    else:
        print(f"Switching ComfyUI version to {VERSION}...")
        !git fetch --all -q
        !git reset --hard {VERSION}
else:
    print("Using latest ComfyUI version.")

if INSTALL_GGUF:
    gguf_dir = f"{BASE_DIR}/custom_nodes/ComfyUI-GGUF"
    if not os.path.exists(gguf_dir):
        print("📥 Installing ComfyUI-GGUF...")
        %cd {BASE_DIR}/custom_nodes
        !git clone https://github.com/city96/ComfyUI-GGUF
        !pip install -r "{gguf_dir}/requirements.txt"
        print(f"✅ ComfyUI-GGUF cloned to {gguf_dir}.")
    else:
        print("✅ ComfyUI-GGUF already exists, skipping.")

print("📥Installing requirements...")
with capture.capture_output() as cap:
    result = os.system("pip install -r requirements.txt")
if result != 0:
    print(cap.stdout)
    raise RuntimeError("pip install failed!")

#@markdown ---
#@markdown ##Colab Notbook Options
CLEAR_OUTPUTS = True #@param {type:"boolean"}
USE_LOGFILE = True #@param {type:"boolean"}
USE_KEEPALIVE = True #@param {type:"boolean"}
#@markdown * **CLEAR_OUTPUTS**: Cleans up colab notebook runtime/install outputs. Leave unchecked for debugging
#@markdown * **USE_LOGFILE**: Writes all 'print' statements to a log file /runtime.log.
#@markdown * **USE_KEEPALIVE**: Starts a long running silent audio player in this cell to help keep session running when idle.

if CLEAR_OUTPUTS:
    clear_output()

print(f"✅ ComfyUI Setup completed in {time.time()-t0:.2f}s")
%cd "{BASE_DIR}"

if USE_KEEPALIVE:
    display(Audio("https://raw.githubusercontent.com/KoboldAI/KoboldAI-Client/main/colab/silence.m4a", autoplay=True))

In [ ]:
#@title #Download and File Tools
# NOTE: aria2c will NOT re-download if file already exists
import os, time, subprocess
from IPython.display import clear_output
from IPython.utils import capture
from urllib.parse import unquote, urlparse, parse_qs
import requests, re

with capture.capture_output() as cap:
    !apt -y install -qq aria2

if CLEAR_OUTPUTS:
    clear_output()

#@markdown # Set Downloader Tokens
CIVITAI_TOKEN = ""  # @param {type:"string"}
HUGFACE_TOKEN = ""  # @param {type:"string"}
#@markdown Not all models require a token, but both CivitAI and Hugging Face do require API tokens for any models that need you to accept a license or TOS before using. If you set a token and the model still fails to download, visit the site and go to the model page and check for a "Click here to accept the agreement" button. Once you accept it, it will be applied to your token and you should be able to re-run and download after that.

LOG_FILE = "/content/runtime.log"

def log(message):
    print(message)
    if USE_LOGFILE:
      timestamp = time.strftime("%Y-%m-%d %H:%M:%S")
      line = f"[{timestamp}] {message}"
      with open(LOG_FILE, "a", encoding="utf-8") as f:
          f.write(line + "\n")

def get_civitai_filename(url: str, api_token: str = None) -> str | None:
    """
    Retrieves the filename for a CivitAI download URL by following the
    redirect and parsing the filename from the destination URL's
    response-content-disposition query parameter.

    Args:
        url: A CivitAI download URL, e.g.
             https://civitai.com/api/download/models/12345
        api_token: Optional CivitAI API token for gated models.

    Returns:
        The filename string, or None if it couldn't be determined.
    """
    headers = {}
    if api_token:
        headers["Authorization"] = f"Bearer {api_token}"
        separator = "&" if "?" in url else "?"
        url = f"{url}{separator}token={api_token}"

    try:
        # Use allow_redirects=False so we can inspect the redirect URL directly
        response = requests.head(url, headers=headers, allow_redirects=False, timeout=10)

        # Civitai responds with a 302 redirect to a pre-signed S3 URL
        redirect_url = response.headers.get("Location", "")

        if redirect_url:
            # Parse the `response-content-disposition` query param from the S3 URL
            parsed = urlparse(redirect_url)
            query_params = parse_qs(parsed.query)
            content_disposition = query_params.get("response-content-disposition", [None])[0]

            if content_disposition:
                content_disposition = unquote(content_disposition)

                # Method 1 — RFC 5987 extended syntax: filename*=UTF-8''my%20model.safetensors
                # This is the preferred modern format. The filename is percent-encoded
                # and prefixed with the charset (usually UTF-8''). We unquote it a
                # second time to handle any percent-encoding in the filename itself,
                # e.g. spaces encoded as %20.
                match = re.search(r"filename\*=(?:UTF-8'')?([^\s;]+)", content_disposition, re.IGNORECASE)
                if match:
                    log("Filename Method 1 - RFC 5987 extended syntax.")
                    return unquote(match.group(1))

                # Method 2 — Quoted filename: filename="my model.safetensors"
                # The traditional format where the filename is wrapped in double
                # quotes. This handles filenames with spaces or special characters
                # that would otherwise break the header parsing.
                match = re.search(r'filename="([^"]+)"', content_disposition)
                if match:
                    log("Filename Method 2 - Traditional quoted filename.")
                    return match.group(1)

                # Method 3 — Bare unquoted filename: filename=mymodel.safetensors
                # The simplest format, used when the filename contains no spaces or
                # special characters. We stop at whitespace or a semicolon since
                # those delimit the next header parameter.
                match = re.search(r"filename=([^\s;]+)", content_disposition)
                if match:
                    log("Filename Method 3 - Bare unquoted filename.")
                    return match.group(1)

        # Method 4 — Fallback: extract filename directly from the S3 URL path.
        # If none of the Content-Disposition methods succeeded (e.g. the parameter
        # was missing entirely), we try the URL path as a last resort. S3 object
        # keys often look like /model/1234/abcd.safetensors, so we take the last
        # path segment. We only accept it if it contains a dot, to avoid returning
        # a bare directory name or ID with no extension.
        if redirect_url:
            path = urlparse(redirect_url).path
            filename = path.split("/")[-1]
            if "." in filename:
                log("Filename Method 4 - Extract from S3 path.")
                return unquote(filename)

    except requests.RequestException as e:
        log(f"Get Filename Request FAILED - returning None: {e}")

    return None


def get_url_filename(url: str, api_token: str = None) -> str | None:
    """
    Extracts a filename from a URL:
    - If the filename is part of the url, return that part only
    - If this is a civitai URL, get filename from content-disposition
    """
    basename = os.path.basename(urlparse(url).path)
    if "." in basename:
        log("Filename Method 0 - Filename is part of url path.")
        return basename
    else:
        return get_civitai_filename(url, api_token)


def download_model(url, dest_dir, filename=None):
    t0 = time.time()
    os.makedirs(dest_dir, exist_ok=True)
    domain = urlparse(url).netloc.lower()

    max_tries = 10
    timeout = 60
    download_url = url

    if CIVITAI_TOKEN and "civitai.com" in domain:
        sep = "&" if "?" in url else "?"
        download_url = f"{url}{sep}token={CIVITAI_TOKEN}"

    # ARIA2C PROCESS
    aria_cmd = [
        "aria2c", "-c", "-x", "16", "-s", "16", "-k", "1M",
        "--retry-wait=5", "--connect-timeout=30",
        f"--max-tries={max_tries}", f"--timeout={timeout}",
        download_url, "-d", dest_dir,
    ]

    if HUGFACE_TOKEN and "huggingface.co" in domain:
        aria_cmd += [f"--header=Authorization: Bearer {HUGFACE_TOKEN}"]

    # debugging use this and then grep the file
    # aria_cmd += ["--log=/tmp/aria2.log", "--log-level=debug"]
    #!grep -E "WARN|ERROR|response code|HTTP" /tmp/aria2.log | tail -30

    if filename:
        aria_cmd += ["-o", filename]
        log(f"📥 Downloading {filename} from {domain}...")
    else:
        aria_cmd += ["--content-disposition"]
        log(f"📥 Downloading model from {domain}...")

    # NOTE sometimes civitai will fail on aria because it does not like multiple connections
    # nothing we can do about it except try it and then fall back to wget
    before = set(os.listdir(dest_dir))
    result = subprocess.run(aria_cmd)

    # WGET PROCESS
    if result.returncode != 0:
        log(f"⚠️ Warning: aria2 failed (code {result.returncode}), falling back to wget...")
        wget_cmd = [
            "wget", "--progress=bar:force", "--continue",
            f"--tries={max_tries}", f"--timeout={timeout}",
        ]

        if HUGFACE_TOKEN and "huggingface.co" in domain:
            wget_cmd += [f"--header=Authorization: Bearer {HUGFACE_TOKEN}"]

        if filename:
            wget_cmd += ["-O", os.path.join(dest_dir, filename), download_url]
        else:
            wget_cmd += ["--content-disposition", "-P", dest_dir, download_url]

        subprocess.run(wget_cmd, check=True)

    after = set(os.listdir(dest_dir))
    new_files = after - before
    if new_files:
        filename = new_files.pop()

    log(f"✅ {filename} downloaded in {time.time()-t0:.2f}s")
    return filename

In [ ]:
#@title #Step 2a: Download Image Models
# NOTE: aria2c will NOT re-download if file already exists
import os, time, gc
from IPython.display import clear_output
from google.colab import drive

t0 = time.time()
model_filename = None
clip_filename = None
vae_filename = None

#@markdown #Model to Download
MODEL_LIST = "Z-Image-Turbo" # @param ["Z-Image-Turbo","Z-Image-Base", "Anima","Qwen-Image","Flux2-Klein"]

#@markdown * Models are always stored in the Runtime Filesystem regardless of ComfyUI location.
#@markdown * Always use the full download URL including any path parameters.
#@markdown * The path parameters determine which FP and size to download.

if MODEL_LIST == "Z-Image-Turbo":
  # Z-Image-Turbo FP8 (5.7GB)
  # FIXED COMFYUI, CFG 2, STEPS 6, euler/simple, SKIP=1 (default)
  model_url = "https://huggingface.co/drbaph/Z-Image-Turbo-FP8/resolve/main/z_image_turbo_fp8_e4m3fn.safetensors"
  # Text Encoder: qwen3_4b_fp8_scaled.safetensors - 3.5GB
  clip_url  = "https://huggingface.co/jiangchengchengNLP/qwen3-4b-fp8-scaled/resolve/main/qwen3_4b_fp8_scaled.safetensors"
  vae_url   = "https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/vae/ae.safetensors"

  if "e4m3fn" in model_url:
      weight_dtype = "fp8_e4m3fn_fast"
  elif "e5m2" in model_url:
      weight_dtype = "fp8_e5m2"
  clip_type = "lumina2"
  log(f"Model zit: {weight_dtype}, {clip_type}")

elif MODEL_LIST == "Z-Image-Base":
  # Z-Image-Base FP8 (6.15GB)
  # LATEST COMFYUI, CFG 4, STEPS 25, euler/simple, SKIP=1 (default)
  model_url = "https://huggingface.co/drbaph/Z-Image-fp8/resolve/main/z-img_fp8-e4m3fn-scaled.safetensors?download=true"
  # Text Encoder: qwen3_4b_fp8_scaled.safetensors - 3.5GB
  clip_url  = "https://huggingface.co/jiangchengchengNLP/qwen3-4b-fp8-scaled/resolve/main/qwen3_4b_fp8_scaled.safetensors"
  vae_url   = "https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/vae/ae.safetensors"

  if "e4m3fn" in model_url:
      weight_dtype = "fp8_e4m3fn"
  elif "e5m2" in model_url:
      weight_dtype = "fp8_e5m2"
  clip_type = "lumina2"
  log(f"Model zib: {weight_dtype}, {clip_type}")

elif MODEL_LIST == "Anima":
  # Anima Preview FP8 (3.89GB)
  # Start promtps with this: masterpiece, best quality, score_7,
  # LATEST COMFYUI, CFG 4, STEPS 30, euler_ancestral/simple, SKIP=0 <<- Important!
  model_url = "https://huggingface.co/Bedovyy/Anima-FP8/resolve/main/anima-preview2-fp8.safetensors?download=true"
  clip_url = "https://huggingface.co/circlestone-labs/Anima/resolve/main/split_files/text_encoders/qwen_3_06b_base.safetensors?download=true"
  vae_url = "https://huggingface.co/circlestone-labs/Anima/resolve/main/split_files/vae/qwen_image_vae.safetensors?download=true"

  weight_dtype = "fp8"
  clip_type = "qwen_image"
  log(f"Model anima: {weight_dtype}, {clip_type}")

elif MODEL_LIST == "Flux2-Klein":
  # Flux 2 Klein 9B FP8 (9.4GB)
  # LATEST COMFYUI, CFG 4, STEPS 20, euler/karras, SKIP 0 or 1 cant see difference
  model_url = "https://huggingface.co/black-forest-labs/FLUX.2-klein-9b-fp8/resolve/main/flux-2-klein-9b-fp8.safetensors"
  clip_url  = "https://huggingface.co/Comfy-Org/vae-text-encorder-for-flux-klein-9b/resolve/main/split_files/text_encoders/qwen_3_8b_fp4mixed.safetensors"
  vae_url = "https://huggingface.co/Comfy-Org/vae-text-encorder-for-flux-klein-9b/resolve/main/split_files/vae/flux2-vae.safetensors"

  weight_dtype = "fp8"
  clip_type = "flux2"

elif MODEL_LIST == "Qwen-Image":
  # Qwen Image (20.4GB, Clip=9.3GB)
  # CFG 2.5, STEPS 20, euler/simple
  # Qwen_Image-Q2_K.gguf - 7GB
  model_url = "https://huggingface.co/QuantStack/Qwen-Image-GGUF/resolve/main/Qwen_Image-Q2_K.gguf"
  weight_dtype = "GGUF"

  # qwen_2.5_vl_7b_fp8_scaled - 9GB (qwen_image)
  clip_url  = "https://huggingface.co/Comfy-Org/Qwen-Image_ComfyUI/resolve/main/split_files/text_encoders/qwen_2.5_vl_7b_fp8_scaled.safetensors?download=true"
  vae_url   = "https://huggingface.co/Comfy-Org/Qwen-Image_ComfyUI/resolve/main/split_files/vae/qwen_image_vae.safetensors?download=true"

  clip_type = "qwen_image"
  log(f"Model Qwen: {weight_dtype}, {clip_type}")

else:
  log("⚠️ WARNING: Invalid Model Type")
  weight_dtype = "fp16"
  clip_type = "lumina2"
  log(f"⚠️ Model UNKNOWN: {weight_dtype}, {clip_type}")

# get the filename from the URL
model_filename = get_url_filename(model_url, CIVITAI_TOKEN)
clip_filename = get_url_filename(clip_url, CIVITAI_TOKEN)
vae_filename = get_url_filename(vae_url, CIVITAI_TOKEN)

# Models will always be stored on the Runtime Filesystem
BASE_MODEL_DIR = "/content/ComfyUI/models"
DIFF_DIR = f"{BASE_MODEL_DIR}/diffusion_models" # ZIT models go in /diffusion_models
os.makedirs(DIFF_DIR, exist_ok=True)
CLIP_DIR = f"{BASE_MODEL_DIR}/clip"
os.makedirs(CLIP_DIR, exist_ok=True)
VAE_DIR = f"{BASE_MODEL_DIR}/vae"
os.makedirs(VAE_DIR, exist_ok=True)

if CLEAR_OUTPUTS:
    clear_output()

# If the URL did not have a filename, we set the name based on the file that gets created in the file system.
# Otherwise we set it to the filename passed in the parameter.
# We need this to be correct because filenames are used in the pipeline setup code.
model_filename = download_model(model_url, DIFF_DIR, model_filename)
clip_filename = download_model(clip_url, CLIP_DIR, clip_filename)
vae_filename = download_model(vae_url, VAE_DIR, vae_filename)
log(f"Cell Runtime: {time.time()-t0:.2f}s")

In [ ]:
#@title #Step 2b: Download LoRAs (Optional)
import os, time
t0 = time.time()

# Make sure we have a folder for the LoRAs
lora_dir = f"{BASE_MODEL_DIR}/loras"
os.makedirs(lora_dir, exist_ok=True)

#@markdown # Select LoRAs to download
DOWNLOAD_LORA_1 = False #@param {type:"boolean"}
LORA_URL_1 = "" #@param {type:"string"}
#@markdown ---
DOWNLOAD_LORA_2 = False #@param {type:"boolean"}
LORA_URL_2 = "" #@param {type:"string"}

if DOWNLOAD_LORA_1:
    # Harcoding the LoRA names is just easier for everything.
    lora_filename_1 = "LoRA_1.safetensors"
    lora_filename_1 = download_model(LORA_URL_1, lora_dir, lora_filename_1)
if DOWNLOAD_LORA_2:
    # Harcoding the LoRA names is just easier for everything.
    lora_filename_2 = "LoRA_2.safetensors"
    lora_filename_2 = download_model(LORA_URL_2, lora_dir, lora_filename_2)

log(f"Cell Runtime: {time.time()-t0:.2f}s")

In [ ]:
#@title #Step 2c: Setup ComfyUI Pipeline
import os, random, time
import numpy as np
import torch, gc, sys
import folder_paths
from PIL import Image

t0 = time.time()
t1 = time.time()

log("Beginning ComfyUI pipeline setup...")
# NOTE: this import takes a LONG (82s) time when ComfyUI on GDrive
from nodes import NODE_CLASS_MAPPINGS
from nodes import load_custom_node
log(f"node import took {time.time()-t1:.2f}s."); t1 = time.time()

# Tell ComfyUI to look in the Runtime Filesystem for the models.
# Runtime files are separate from GDrive.
sys.path.insert(0, BASE_DIR)
folder_paths.add_model_folder_path("diffusion_models", f"{BASE_MODEL_DIR}/diffusion_models")
folder_paths.add_model_folder_path("checkpoints",      f"{BASE_MODEL_DIR}/checkpoints")
folder_paths.add_model_folder_path("clip",             f"{BASE_MODEL_DIR}/clip")
folder_paths.add_model_folder_path("vae",              f"{BASE_MODEL_DIR}/vae")
folder_paths.add_model_folder_path("loras",            f"{BASE_MODEL_DIR}/loras")

# VRAM GC allows re-entrant calls to this step.
if 'unet' in globals():
    del unet
if 'clip' in globals():
    del clip
if 'vae' in globals():
    del vae
gc.collect()
torch.cuda.empty_cache()
log(f"🧹 Cleaning VRAM took {time.time()-t1:.2f}s."); t1 = time.time()

# Setup ComfyUI Nodes
UNETLoader       = NODE_CLASS_MAPPINGS["UNETLoader"]()
CLIPLoader       = NODE_CLASS_MAPPINGS["CLIPLoader"]()
VAELoader        = NODE_CLASS_MAPPINGS["VAELoader"]()
LoraLoader       = NODE_CLASS_MAPPINGS["LoraLoader"]()
CLIPTextEncode   = NODE_CLASS_MAPPINGS["CLIPTextEncode"]()
EmptyLatentImage = NODE_CLASS_MAPPINGS["EmptyLatentImage"]()
KSampler         = NODE_CLASS_MAPPINGS["KSampler"]()
VAEDecode        = NODE_CLASS_MAPPINGS["VAEDecode"]()
# this is for clipskip setting
CLIPSetLastLayer = NODE_CLASS_MAPPINGS["CLIPSetLastLayer"]()
# this is for working with gguf files
if INSTALL_GGUF:
    GGUFLoader = NODE_CLASS_MAPPINGS["UnetLoaderGGUF"]()

with torch.inference_mode():
    log("Loading models into VRAM...")
    if INSTALL_GGUF and weight_dtype == "GGUF":
        unet = GGUFLoader.load_unet(model_filename)[0]
        log(f"GGUFLoader({model_filename}) took {time.time()-t1:.2f}s."); t1 = time.time()
    else:
        unet = UNETLoader.load_unet(model_filename, weight_dtype)[0]
        log(f"UNETLoader({model_filename}, {weight_dtype}) took {time.time()-t1:.2f}s."); t1 = time.time()
    clip = CLIPLoader.load_clip(clip_filename, type=clip_type)[0]
    log(f"CLIPLoader({clip_filename}, {clip_type}) took {time.time()-t1:.2f}s."); t1 = time.time()
    vae = VAELoader.load_vae(vae_filename)[0]
    log(f"VAELoader({vae_filename}) took {time.time()-t1:.2f}s."); t1 = time.time()


# BEGIN NEW METHOD
# Bake LoRAs into the model at load time (avoids per-generation RAM overhead)
LORA1 = 0 #@param {type:"slider", min:0.0, max:2.0, step:0.01}
LORA2 = 0 #@param {type:"slider", min:0.0, max:2.0, step:0.01}
log("Loading LoRAs...")
if 'lora_filename_1' in globals() and LORA1 > 0:
    log(f"Applying LoRA 1 at strength {LORA1}...")
    unet, clip = LoraLoader.load_lora(unet, clip, lora_filename_1, LORA1, LORA1)
    log(f"✅ LoRA 1 applied.")
if 'lora_filename_2' in globals() and LORA2 > 0:
    log(f"Applying LoRA 2 at strength {LORA2}...")
    unet, clip = LoraLoader.load_lora(unet, clip, lora_filename_2, LORA2, LORA2)
    log(f"✅ LoRA 2 applied.")

gc.collect()
torch.cuda.empty_cache()
# END NEW METHOD

# Safety checks
if unet is None:
    log("⚠️ Model missing UNET!")
if clip is None:
    log("⚠️ Model missing CLIP!")
if vae is None:
    log("⚠️ Model missing VAE!")
log(f"✅ Pipeline setup completed in {time.time()-t0:.2f}s")

In [ ]:
#@title #Step 2d: Define the Generate Function
import time, gc

@torch.inference_mode()
def generate(input):
    values          = input["input"]
    positive_prompt = values['positive_prompt']
    negative_prompt = values['negative_prompt']
    seed            = values['seed']
    steps           = values['steps']
    cfg             = values['cfg']
    sampler_name    = values['sampler_name']
    scheduler       = values['scheduler']
    denoise         = values['denoise']
    width           = values['width']
    height          = values['height']
    batch_size      = values['batch_size']
    clip_skip       = values.get('clip_skip', 1)        # default = no skip
    #lora_strengths  = values.get('lora_strengths', [])  # list of (filename, strength)

    t0 = time.time()

    if seed is None or seed == 0:
        random.seed(int(time.time()))
        seed = random.randint(0, 18446744073709551615)

    # Apply LoRAs if any are active — starts from base model each request
#    model = unet
#    clip_model = clip
#    lora_was_applied = False
#    for lora_file, strength in lora_strengths:
#        if strength > 0:
#            model, clip_model = LoraLoader.load_lora(model, clip_model, lora_file, strength, strength)
#            lora_was_applied = True

    #clip_model_skipped = CLIPSetLastLayer.set_last_layer(clip_model, stop_at_clip_layer=-clip_skip)[0]
    clip_model_skipped = CLIPSetLastLayer.set_last_layer(clip, stop_at_clip_layer=-clip_skip)[0]
    positive           = CLIPTextEncode.encode(clip_model_skipped, positive_prompt)[0]
    negative           = CLIPTextEncode.encode(clip_model_skipped, negative_prompt)[0]
    latent_image       = EmptyLatentImage.generate(width, height, batch_size=batch_size)[0]
    #samples            = KSampler.sample(model, seed, steps, cfg, sampler_name, scheduler, positive, negative, latent_image, denoise=denoise)[0]
    samples            = KSampler.sample(unet, seed, steps, cfg, sampler_name, scheduler, positive, negative, latent_image, denoise=denoise)[0]
    decoded            = VAEDecode.decode(vae, samples)[0].detach()

    # Get the images (if batch > 1 we get an array)
    images = [
      Image.fromarray(np.array(frame * 255, dtype=np.uint8))
      for frame in decoded
    ]

    # Explicit cleanup
    del decoded, samples, latent_image, positive, negative, clip_model_skipped
#    if lora_was_applied:
#        del model
#        del clip_model
    gc.collect()
    torch.cuda.empty_cache()

    elapsed = time.time() - t0
    return images, seed, elapsed

In [ ]:
#@title #Test: Generate Image Locally
#@markdown # Image Generation Settings
#@markdown This image generator cell calls generate() directly and does not need the Web API to be running.
import time
from IPython.display import HTML, display

POSITIVE_PROMPT = "" #@param {type:"string"}
#@markdown
NEGATIVE_PROMPT = "" #@param {type:"string"}
#@markdown
#@markdown * Z-Image-Turbo: Fixed, CFG 1, STEPS 6, euler/simple, SKIP=1 (default)
#@markdown * Z-Image-Base: Latest, CFG 4, STEPS 25, euler/simple, SKIP=1 (default)
#@markdown * Anima: Latest, CFG 4, STEPS 30, euler_ancestral/simple, SKIP=0 <<- Important!
#@markdown * Flux Klein: Latest, CFG 4, STEPS 20, euler/karras, SKIP 0 or 1 cant see difference

CFG = 4 #@param {type:"number"}
STEPS = 20 # @param {"type":"integer"}
CLIP_SKIP = 1 # @param {"type":"integer"}
SAMPLER = "euler" #@param {type:"string"}
SCHEDULER = 'karras' #@param ['beta', 'ddim_uniform', 'exponential', 'karras', 'kl_optimal', 'linear_quadratic', 'normal', 'sgm_uniform', 'simple']
SEED = 0 # @param {"type":"integer"}
#DENOISE = 1.0 # @param {"type":"number"}
WIDTH = 800 # @param {"type":"integer"}
HEIGHT = 600 # @param {"type":"integer"}
#DENOISE = 1 #@param {type:"slider", min:0.0, max:1.0, step:0.01}
DENOISE = 1

BATCHES = 1 # @param {"type":"integer"}
#LORA1 = 0.8 #@param {type:"slider", min:0.0, max:2.0, step:0.01}
#LORA2 = 0 #@param {type:"slider", min:0.0, max:2.0, step:0.01}
SAVE_IMAGES = False # @param {type:"boolean"}

#@markdown * **DPM++ 2M Karras**: dpmpp_2m + karras
#@markdown * **DPM++ SDE Karras**: dpmpp_sde + karras
#@markdown * **DPM++ 2M SDE Exponential**: dpmpp_2m_sde + exponential
#@markdown * **Euler a**: euler_ancestral + normal
#@markdown ---
#@markdown ### Valid Samplers
#@markdown 'euler', 'euler_cfg_pp', 'euler_ancestral', 'euler_ancestral_cfg_pp', 'heun', 'heunpp2', 'dpm_2', 'dpm_2_ancestral', 'lms', 'dpm_fast', 'dpm_adaptive', 'dpmpp_2s_ancestral', 'dpmpp_2s_ancestral_cfg_pp', 'dpmpp_sde', 'dpmpp_sde_gpu', 'dpmpp_2m', 'dpmpp_2m_cfg_pp', 'dpmpp_2m_sde', 'dpmpp_2m_sde_gpu', 'dpmpp_2m_sde_heun', 'dpmpp_2m_sde_heun_gpu', 'dpmpp_3m_sde', 'dpmpp_3m_sde_gpu', 'ddpm', 'lcm', 'ipndm', 'ipndm_v', 'deis', 'res_multistep', 'res_multistep_cfg_pp', 'res_multistep_ancestral', 'res_multistep_ancestral_cfg_pp', 'gradient_estimation', 'gradient_estimation_cfg_pp', 'er_sde', 'seeds_2', 'seeds_3', 'sa_solver', 'sa_solver_pece', 'ddim', 'uni_pc', 'uni_pc_bh2'

if not POSITIVE_PROMPT:
    POSITIVE_PROMPT = "a beautiful woman in a red bikini walking on a white sandy beach"
if not NEGATIVE_PROMPT:
    NEGATIVE_PROMPT = "bad quality, worst quality, weird anatomy"

request = {
    "positive_prompt": POSITIVE_PROMPT,
    "negative_prompt": NEGATIVE_PROMPT,
    "width": WIDTH, "height": HEIGHT,
    "denoise": DENOISE, "batch_size": BATCHES,
    "steps": STEPS, "cfg": CFG, "clip_skip": CLIP_SKIP,
    "sampler_name": SAMPLER, "scheduler": SCHEDULER,
    "seed": SEED,         # 0 = random seed
    "lora_strengths": [("LoRA_1.safetensors", LORA1),("LoRA_2.safetensors", LORA2)]
}

# Build the input payload expected by generate()
input_payload = {
    "input": {
        **request,
        "sampler_name": SAMPLER,
        "scheduler": SCHEDULER,
    }
}

#log(f"📤 Request:\n{json.dumps(request, indent=2)}\n")
images, used_seed, elapsed = generate(input_payload)

log(f"Model: {model_filename}: {WIDTH}x{HEIGHT} | {CFG},{STEPS},{CLIP_SKIP} | {SAMPLER}/{SCHEDULER} | {elapsed:.1f}s | Seed: {used_seed}")
for img in images:
    display(img)

# Save image if checkbox is enabled
if SAVE_IMAGES:
    save_dir = "/content/SavedImages"
    os.makedirs(save_dir, exist_ok=True)
    for i, img in enumerate(images):
      filename = f"img_seed_{used_seed}_{int(time.time())}_{i}.png"
      filepath = os.path.join(save_dir, filename)
      img.save(filepath)
      log(f"💾 Image saved to: {filepath}")